# The check, then lakefed's own ingest runs

Runs first, on serverless, over one group of collections (`task_collections`: a schedule's members, or
blank for every collection with enabled rows), and does what a sweep's planning does first
(`initial_check.run_check`): each database is asked whether anything of ours changed. A database with
nothing to do gets its check stamp (`database_check`) and no ingest.

Then it starts the ingestion job itself (`launcher.py`): for each collection with work, it records the
databases with work in `check_work` and starts one run of `lakefed_ingest` with `run-now`
(`task_collection` and that list's `work_id`), the way the operations app's Run now starts it. No work,
no run: no ingest cluster starts. Inside each run nothing changes: its tables run on threads and
scheduler pools.

- **Start and end** (default): it starts the runs, watches them for `confirm_s` seconds, and ends.
  A run Databricks skips (the ingestion job at its `max_concurrent_runs`, queue off), that fails at
  once, or that no poll could read in that time (unconfirmed) fails this task, named. Each run reports its own result after that. It starts no more runs than
  the ingestion job has room for (with its queue on, all: the queue holds the rest); the rest are named
  and wait for the next cycle. With `wait` true it
  polls every run until all have ended instead (`max_running`, `poll_s`, `timeout_s`), billed for the
  whole time.
- A collection whose run is still going (from an earlier cycle) is not started again, and a collection
  whose lock a sweep holds is left to that sweep.
- **Network**: the check reads the source databases itself, from serverless compute, so serverless needs a
  network path to every source (for example private endpoints through a network connectivity
  configuration, or the sources' firewall rules). A source only a job cluster can reach can't be checked.
- **Source cap**: with `max_queries_per_source` set, the check's own source queries count under it, as an
  ingest run's do: it takes its share of each server's cap, so checks and runs together stay within it.
- **Identity**: the Jobs API calls are made as this job's run-as identity, which needs
  `CAN_MANAGE_RUN` on the ingestion job (`ingest_job_id`). Each run runs as the ingestion job's own
  run-as identity.
- **Packages**: `psycopg` and the Databricks SDK come from the job's environment
  (`resources/lakefed_ingest_check.yml`), not a `%pip` here.

In [ ]:
# Bootstrap: put this notebook's dir on sys.path and import the shared modules.
import sys, os, json, uuid
from concurrent.futures import ThreadPoolExecutor

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import state_store as ss
from state_store import StateStore
import ct_engine
import initial_check
import launcher
import planner

In [ ]:
def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


def _safe(fn, default=""):
    try:
        return fn() or default
    except Exception:
        return default


task_collections = [c.strip() for c in widget("task_collections", "").split(",") if c.strip()]
# An app ingestion job's id: its groups are read from Lakebase, since a long list of groups doesn't fit in one
# job parameter (task_collections is then left blank, or holds the same groups).
schedule_id = widget("schedule_id", "").strip()
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
# Databases checked at once (one source query each; a changed database runs its batched detection).
check_parallelism = int(widget("check_parallelism", "32"))
detection_chunk = int(widget("detection_chunk", str(ct_engine.DEFAULT_DETECTION_CHUNK)))
# The same meaning as in ingest.ipynb.
freshness_interval_s = float(widget("freshness_interval_s", "0"))
reseed_on_new_columns = widget("reseed_on_new_columns", "true").strip().lower() in ("1", "true", "yes")
os.environ["LAKEFED_SQL_SECRET_SCOPE"] = widget("secret_scope", "lakefed_ingest")
os.environ["LAKEFED_SQL_TRUST_SERVER_CERT"] = widget("trust_server_certificate", "true")
os.environ["LAKEFED_SQL_SNAPSHOT_ISOLATION"] = widget("snapshot_isolation", "false")
force = {d.strip() for d in widget("force_databases", "").split(",") if d.strip()}
# The start: the ingestion job. The check waits for its runs (wait) and ends with them, failing when one failed; the
# wait is bounded by timeout_s (0: no limit), after which the runs still going are cancelled and the check fails.
# Off, it only watches the runs for confirm_s.
ingest_job_id = int(widget("ingest_job_id", "0"))
wait = widget("wait", "true").strip().lower() in ("1", "true", "yes")
confirm_s = float(widget("confirm_s", "30"))
max_running = int(widget("max_running", "0"))
poll_s = float(widget("poll_s", "30"))
timeout_s = float(widget("timeout_s", "21600"))
# Ingestion job parameters for every run: the check's own settings above, plus the run-only ones
# below when set (blank keeps the job's default).
for name in ("parallelism", "append_batch_size", "max_seeds_per_run", "max_queries_per_source",
             "table_retries", "table_retry_budget", "fail_on_table_errors", "spark_conf"):
    widget(name, "")
ct_engine.spark_conf_settings(dbutils.widgets.get("spark_conf"))   # a bad setting fails here, before any run starts
# The check's own source queries count under the same per-server cap as the runs' (initial_check.SourceCap).
max_queries_per_source = int(dbutils.widgets.get("max_queries_per_source") or 0)
run_parameters = launcher.run_parameters({name: dbutils.widgets.get(name) for name in launcher.RUN_PARAMETERS})
assert ingest_job_id > 0, "ingest_job_id is required: the ingestion job to start"
# How the cycle's runs are made (resources/lakefed_ingest_check.yml): by the changes (groups B), or one per group.
runs_by = widget("runs_by", "group").strip().lower() or "group"
assert runs_by in ("changes", "group"), f"runs_by must be changes or group, not {runs_by}"

job_run_id = _safe(lambda: str(ctx.jobRunId().get()))
# This attempt of the task (its task run id, from the job's {{task.run_id}}): one start and one work
# list per collection per attempt, whatever the start call's own retries do.
# Run by hand, outside a job, it has neither: a fresh id, so Databricks never hands back an earlier run for it.
launch_id = widget("launch_id", "") or job_run_id or f"manual-{uuid.uuid4().hex[:12]}"
cluster_id = _safe(lambda: spark.conf.get("spark.databricks.clusterUsageTags.clusterId"), "serverless")
store = StateStore.connect_lakebase(db_instance, db_name, reconnect=True)

# A Lakebase behind this code's schema version is migrated here, as an ingest run would.
_schema_have = store.schema_version()
if _schema_have < ss.SCHEMA_VERSION:
    with open("/Workspace" + nb_dir + "/lakebase_schema.sql") as f:
        schema_sql = f.read()
    try:
        _migrated = store.migrate(schema_sql)
    except ss.MigrationLockTimeout as e:
        print(f"[schema] the migration of the Lakebase schema from version {_schema_have} to "
              f"{ss.SCHEMA_VERSION} waits: a running load holds its tables. Nothing is checked or started; "
              f"the next check migrates. {str(e)[:300]}")
        dbutils.notebook.exit(json.dumps({"collections": 0, "status": "migration_waiting",
                                          "schema_version": _schema_have}))
    if _migrated:
        print(f"migrated the Lakebase schema from version {_migrated['from']} to {ss.SCHEMA_VERSION}")
# A login typed into the operations app goes only to the host of the connection it was typed for
# (ct_engine.typed_login_refusal), whatever a control row or a discovery spec names.
_login_scope = os.environ["LAKEFED_SQL_SECRET_SCOPE"]
ct_engine.guard_typed_logins(_login_scope, store.typed_logins(_login_scope))

if schedule_id:
    task_collections = store.schedule_collections(int(schedule_id))
elif not task_collections:
    task_collections = store.enabled_collections()
else:
    deleted_only = set(store.deleted_only_collections())
    skipped = [c for c in task_collections if c in deleted_only]
    if skipped:
        print(f"skipping {len(skipped)} group(s) only a deleted ingestion job holds: {', '.join(skipped)}")
    task_collections = [c for c in task_collections if c not in deleted_only]
if not task_collections:   # an ingestion job with no groups (or deleted), or no enabled rows: nothing to check
    print("no collections to check" + (f" in ingestion job {schedule_id}" if schedule_id else "") + ", so no run starts")
    dbutils.notebook.exit(json.dumps({"collections": 0}))
lock_store = StateStore.connect_lakebase(
    db_instance, db_name, application_name=ss.lock_application_name("check", job_run_id))
print(f"checking {len(task_collections)} collection(s) (check_parallelism={check_parallelism}, "
      f"force={sorted(force) or '-'}), then starting job {ingest_job_id} for each with work (wait={wait})")

In [ ]:
# Check every collection of the group; databases are checked in parallel across all of them,
# through one shared pool. A collection with work keeps its lock until its run is started (cell below).
pool = ThreadPoolExecutor(max_workers=max(1, check_parallelism), thread_name_prefix="check")
# With a per-server cap, the check joins each server's shared lock as a run does, on a connection of its own
# that holds them until the check ends, and takes its share of the cap.
source_store, source_cap = None, None
if max_queries_per_source > 0:
    source_store = StateStore.connect_lakebase(
        db_instance, db_name, application_name=ss.lock_application_name("check-sources", job_run_id))
    source_cap = initial_check.SourceCap(max_queries_per_source, source_store.join_sources, source_store.source_runs,
                                         ct_engine.source_key)


def detect_one(members):
    return planner.detect_group(
        members, current_version=None,
        database_state=lambda c: ct_engine.database_ct_state(c, spark=spark),
        detect=lambda c, items: ct_engine.detect_changes(spark, c, items, chunk_size=detection_chunk))


try:
    summary = initial_check.run_check(
        store, lock_store, task_collections, detect_one, job_run_id=job_run_id, cluster_id=cluster_id,
        existing_sinks_of=lambda configs: ct_engine.existing_sinks(spark, planner.sink_checked_configs(configs)),
        parallelism=check_parallelism, interval_s=freshness_interval_s, reseed_on_new_columns=reseed_on_new_columns,
        snapshot_isolation=ct_engine.snapshot_isolation(), force=force, map_fn=pool.map, source_cap=source_cap,
        hold_work_locks=runs_by == "group", share_locks=runs_by == "changes")
except BaseException:
    lock_store.close()     # releases every lock it holds
    raise
finally:
    pool.shutdown(wait=True)
    if source_store is not None:
        source_store.close()
if summary["failed"]:
    print(initial_check.failure_message(summary["failed"]))

In [ ]:
# Record each collection's work list, then start one ingestion run per collection with work. Each collection's
# lock, held since its check, is let go right after its start call (or at once when it won't be started).
held = set(summary["held"])


def release(collection):
    if collection in held:
        held.discard(collection)
        initial_check.release_locks(lock_store, [collection])


try:
    jobs = launcher.SdkJobs()
    if runs_by == "changes":
        # The runs going have their databases (a run of a share has its whole work list): those wait for them.
        active = jobs.active_runs(ingest_job_id) if any(r["work"] for r in summary["per_collection"]) else []
        running = launcher.active_collections(active, work_of=store.get_check_work,
                                              work_all_of=store.get_check_work_all)
        # The job's own tables per group, else its tables over its groups.
        group_size = ((store.schedule_tables_per_group(int(schedule_id)) if schedule_id else None)
                      or initial_check.group_size_of({r["collection"]: r["tables"] for r in summary["per_collection"]}))
        cycle = launcher.cycle_requests(summary["per_collection"], group_size, parameters=run_parameters,
                                        launch_id=launch_id, running=running)
        requests, already = cycle["requests"], {}
        summary["cycle"] = {"group_size": group_size, "runs": len(requests), "deferred": cycle["deferred"],
                            "shares": [{k: r[k] for k in ("collection", "collections", "databases", "tables",
                                                          "first_loads")} for r in requests]}
        for request in requests:
            for collection, work in request["work"].items():
                store.record_check_work(request["work_id"], collection, work)
    else:
        requests = launcher.run_requests(summary["per_collection"], parameters=run_parameters, launch_id=launch_id)
        for request in requests:
            store.record_check_work(request["work_id"], request["collection"], request["work"])
        active = jobs.active_runs(ingest_job_id) if requests else []
        already = launcher.active_collections(active, work_of=store.get_check_work)
    summary["work_lists_pruned"] = store.prune_check_work(7)
    room = (launcher.capacity(jobs.max_concurrent_runs(ingest_job_id), len(active), jobs.queue_enabled(ingest_job_id))
            if requests else None)
    print(f"{len(requests)} collection(s) with work: "
          + (", ".join(f"{r['collection']} ({r['databases']} db)" for r in requests) or "none, so no run starts")
          + (f"; already running: {sorted(already)}" if already else ""))
    result = launcher.launch(jobs, ingest_job_id, requests, launch_id=launch_id, wait=wait, confirm_s=confirm_s,
                             max_running=max_running, poll_s=poll_s, timeout_s=timeout_s, already_running=already,
                             room=room, on_settled=release)
finally:
    initial_check.release_locks(lock_store, sorted(held))
    lock_store.close()
summary["launch"] = result
print(json.dumps(summary, indent=2, default=str))
failed = launcher.failures(result)
if failed or summary["failed"]:
    raise RuntimeError("\n".join(([launcher.failure_message(failed)] if failed else [])
                                 + ([initial_check.failure_message(summary["failed"])] if summary["failed"] else [])))
dbutils.notebook.exit(json.dumps({k: summary[k] for k in summary if k != "per_collection"}, default=str))